<a href="https://colab.research.google.com/github/95697196-ifts24/restaurador-tpi2/blob/main/notebooks/ejecutar_en_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ampliador de fotos familiares · ejecución en Google Colab

Mi computadora tiene una GPU de **2 GB** (GeForce MX230): no alcanza para el modelo de difusión
(`stabilityai/stable-diffusion-x4-upscaler`, ~3 GB). Por eso, como permite la consigna, lo ejecuto en Colab.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4 → Guardar*.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Descargar el proyecto e instalar dependencias
Clona el repositorio del proyecto desde GitHub.

In [ ]:
# Se puede ejecutar varias veces: clona solo la primera vez y después actualiza.
%cd /content
REPO = "https://github.com/95697196-ifts24/restaurador-tpi2.git"
![ -d restaurador-tpi2 ] || git clone -q {REPO}
%cd /content/restaurador-tpi2
!git pull -q
!pip install -q -r requirements-colab.txt

## 2. Opción A · Abrir la aplicación

Ejecutá la celda y **abrí el link `https://….gradio.live`** que aparece.

> ⏳ **Esta celda no termina nunca, y está bien:** enciende un servidor que queda esperando
> que alguien use la app (el ícono ■ sigue girando). El progreso de cada ampliación se ve
> **en la página de la app**, no acá. La **primera** ampliación descarga el modelo (~3 GB, varios minutos).
>
> Cuando termines, apretá ■ para apagar el servidor.

In [ ]:
!python app.py --share

## 3. Opción B · Prueba de laboratorio: ¿cuánto se acercó la IA a la verdad?

Con una foto real dañada **no hay referencia** para comparar. Acá hacemos trampa a propósito:
partimos de una foto buena (`ejemplos/referencia/`), la **achicamos 4 veces**, la ampliamos con bicúbica y con IA,
y comparamos contra la original. Así sí podemos **verificar** qué inventó el modelo.

*(Si la app de la Opción A está corriendo, detenela antes con ■.)*

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from restaurador import clasico, difusion, textos

NOMBRE = "locomotora_con_numero"   # probá también: cafe, vereda_con_gente
referencia = Image.open(f"ejemplos/referencia/{NOMBRE}.jpg").convert("RGB")
chica = referencia.resize((referencia.width // 4, referencia.height // 4), Image.Resampling.BOX)
bicubica = clasico.ampliar_bicubica(chica)

resultados = {"Bicúbica (clásico)": bicubica}
for nivel, ruido in textos.NIVELES.items():
    resultados[f"IA · {nivel}"] = difusion.ampliar_con_ia(chica, ruido, semilla=1234)
resultados["IA · Poco · otra semilla"] = difusion.ampliar_con_ia(chica, textos.NIVELES[textos.NIVEL_POR_DEFECTO], semilla=99)

print(f"PSNR contra la foto REAL (más alto = más parecido):")
for nombre, imagen in resultados.items():
    print(f"  {nombre:40s} {clasico.psnr(referencia, imagen):5.1f} dB")

In [ ]:
# Vista completa y zoom a una zona con detalle (ajustá la caja si usás otra foto)
CAJA = (130, 120, 330, 320)   # (izquierda, arriba, derecha, abajo) en la imagen de 640 px
paneles = {"REAL (referencia)": referencia, "Entrada chica": chica.resize(referencia.size, Image.Resampling.NEAREST), **resultados}
fig, ejes = plt.subplots(2, len(paneles), figsize=(4 * len(paneles), 8))
for i, (titulo, imagen) in enumerate(paneles.items()):
    imagen = imagen.resize(referencia.size)
    ejes[0, i].imshow(imagen); ejes[0, i].set_title(titulo, fontsize=9)
    ejes[1, i].imshow(imagen.crop(CAJA)); ejes[1, i].set_title("zoom", fontsize=9)
for eje in ejes.ravel(): eje.axis("off")
plt.tight_layout(); plt.savefig("comparacion_laboratorio.png", dpi=110); plt.show()

## 4. Qué anotar para el README y el video

- ¿La IA tiene **más o menos PSNR** que la bicúbica? (Una imagen más nítida puede estar **más lejos** de la verdad.)
- En el zoom: ¿los números/letras/caras que dibujó la IA **coinciden con la foto real**?
- ¿Qué cambió entre la semilla 1234 y la 99? Eso es detalle **inventado** con seguridad.
- Descargá `comparacion_laboratorio.png` (panel de archivos a la izquierda) y subila a `docs/capturas/`.